# 🍎 Responses API with AIProjectClient 🍏

In this notebook, we'll demonstrate how to generate conversational answers using the **Azure AI Foundry** SDK. We'll use the **`azure-ai-projects`** package and the Foundry project's **OpenAI-compatible client** (via `get_openai_client()`) to:

1. **Initialize** an `AIProjectClient`.
2. **Obtain** the project's OpenAI client to do direct LLM calls.
3. **Use** a **prompt template** to add system context.
4. **Send** user prompts in a health & fitness theme through the Responses API.

## 🏋️ Health-Fitness Disclaimer
> **This example is for demonstration only and does not provide real medical advice.** Always consult a professional for health or medical-related questions.

### Prerequisites
Before starting this notebook, please ensure you have completed all prerequisites listed in the root [README.md](../../README.md#-prerequisites).

Let's get started! 🎉

<img src="./seq-diagrams/1-chat.png" width="30%"/>


## 1. Initial Setup
Load environment variables, create an endpoint-based `AIProjectClient`, and get its **OpenAI client** (`get_openai_client()`) for Responses API calls. You'll also define a **prompt template** to show how you might structure system instructions.


In [9]:
import os
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Model deployment name (from Foundry > Models + endpoints)
model_deployment = os.environ.get("MODEL_DEPLOYMENT_NAME")

try:
    # Endpoint-based client (New Foundry) + its OpenAI client for Responses API
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully created AIProjectClient + OpenAI client")
except Exception as e:
    print("❌ Error initializing client:", e)

✅ Successfully created AIProjectClient + OpenAI client


### Prompt Template
We'll define a quick **system** message that sets the context as a friendly, disclaimer-providing fitness assistant.

```txt
SYSTEM PROMPT (template):
You are FitChat GPT, a helpful fitness assistant.
Always remind users: I'm not a medical professional.
Be friendly, provide general advice.
...
```

We'll then pass user content as a **user** message.


In [11]:
# We'll define a function that runs Responses API calls with system instructions and user input
def chat_with_fitness_assistant(user_input: str):
    """Use Responses API to get a response from our LLM, with system instructions."""
    # Our instruction template
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    # Call Responses API via the OpenAI client (instructions + user input)
    response = openai_client.responses.create(
        model=model_deployment,
        instructions=system_text,
        input=user_input,
    )

    return response.output_text

print("Defined a helper function to use the Responses API.")

Defined a helper function to use the Responses API.


## 2. Try Responses API 🎉
We'll call the function with a user question about health or fitness and see the result. Feel free to modify the question or run multiple times!


In [12]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

🗣️ User: How can I start a beginner workout routine at home?
🤖 Assistant: A simple way to start is to keep it short, consistent, and beginner-friendly. I’m not a medical professional, but here’s a safe general approach for many people:

### 1. Start with 3 days per week
Aim for **20–30 minutes per session** on non-consecutive days, like:
- Monday
- Wednesday
- Friday

### 2. Warm up for 5 minutes
Try:
- Marching in place
- Arm circles
- Shoulder rolls
- Gentle bodyweight squats
- Light stretching or mobility moves

### 3. Do a basic full-body routine
Start with **1–2 rounds** of the following:
- **Bodyweight squats** – 8 to 12 reps
- **Wall push-ups** or **knee push-ups** – 6 to 10 reps
- **Glute bridges** – 10 to 15 reps
- **Bird-dogs** – 8 reps per side
- **Standing calf raises** – 10 to 15 reps
- **Plank** or **incline plank** – 10 to 20 seconds

Rest **30–60 seconds** between exercises as needed.

### 4. Add light cardio
After the workout or on separate days, try:
- Walking
- March

## 3. Another Example: Prompt Template with Fill-Ins 📝
We can go a bit further and add placeholders in the system message. For instance, imagine we have a **userName** or **goal**. We'll show a minimal example.


In [13]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    # Construct instruction template with placeholders
    system_template = (
        "You are FitChat GPT, an AI personal trainer for {name}.\n"
        "Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )

    # Fill in placeholders
    system_prompt = system_template.format(name=user_name, goal=goal)

    response = openai_client.responses.create(
        model=model_deployment,
        instructions=system_prompt,
        input=user_input,
    )

    return response.output_text

# Let's try it out
templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance"
)
print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)

🗣️ User: What kind of home exercise do you recommend for a busy schedule?
🤖 Assistant: For a busy schedule, I’d recommend **short, efficient home workouts** that build **muscle tone and endurance** without needing much equipment.

I’m not a medical professional, but here’s a practical approach that works well for a lot of people:

## Best option: 15–20 minute full-body circuits
These are great because they:
- save time
- raise your heart rate
- strengthen multiple muscle groups
- improve endurance

## Simple home workout plan
Try this **3–5 times per week**:

### Circuit example
Do each exercise for **40 seconds**, then rest **20 seconds**.  
Complete **3 rounds**.

1. **Bodyweight squats**
2. **Push-ups**  
   - modify on knees or against a counter if needed
3. **Glute bridges**
4. **Plank**
5. **Reverse lunges**
6. **Mountain climbers**

Rest **1 minute between rounds**.

## If you only have 10 minutes
Do this quick routine:
- 20 squats
- 10 push-ups
- 20 alternating lunges
- 30-seco

## 🎉 Congratulations!
You've successfully used the **Responses API** with Azure AI Foundry's `AIProjectClient` and its OpenAI-compatible client (`get_openai_client()`). You've also seen how to incorporate **prompt templates** to tailor your system instructions.

#### Head to [2-embeddings.ipynb](2-embeddings.ipynb) for the next part of the workshop! 🎯